# 04 — Triangulos de Desarrollo y Reservas IBNR

## La Pieza Central del Analisis Actuarial

Los triangulos de desarrollo son la herramienta fundamental para estimar reservas IBNR (Incurred But Not Reported). Este notebook demuestra:

1. Construccion de triangulos de perdidas incurridas y pagadas
2. Calculo de factores age-to-age (link ratios)
3. Metodo Chain-Ladder paso a paso
4. Metodo Bornhuetter-Ferguson
5. Comparacion CL vs BF vs ultimates conocidos

### ¿Por que importa?

Las aseguradoras deben mantener reservas suficientes para pagar siniestros futuros. Subestimar = insolvencia. Sobreestimar = capital inmovilizado innecesariamente. El triangulo de desarrollo permite proyectar cuanto falta por pagar.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

VALUATION_YEAR = 1997

triangles = pd.read_parquet('../data/processed/triangles.parquet')
ibnr = pd.read_parquet('../data/processed/ibnr_results.parquet')
lob_summary = pd.read_parquet('../data/processed/lob_summary.parquet')

print(f'Triangulos: {len(triangles):,} rows')
print(f'LOBs: {triangles["line_of_business"].unique().tolist()}')

## 1. Visualizacion del Triangulo de Perdidas Pagadas

Un triangulo de desarrollo muestra las perdidas acumuladas por ano de accidente (filas) y lag de desarrollo (columnas). Solo la esquina superior izquierda esta disponible — la inferior derecha es lo que necesitamos estimar.

In [ ]:
# Build paid loss triangle for a specific LOB (as-of valuation date)
LOB = 'Private Passenger Auto'

lob_data = triangles[
    (triangles['line_of_business'] == LOB)
    & (triangles['DevelopmentYear'] <= VALUATION_YEAR)
]

paid_tri = lob_data.pivot_table(
    index='AccidentYear', columns='DevelopmentLag',
    values='CumPaidLoss', aggfunc='sum'
)

print(f'Triangulo de Perdidas Pagadas — {LOB}')
print(f'Shape: {paid_tri.shape} (years x lags)')
print()
(paid_tri / 1_000_000).round(1)  # Display in millions

In [ ]:
# Heatmap visualization
fig = go.Figure(data=go.Heatmap(
    z=paid_tri.values / 1_000_000,
    x=[f'Lag {c}' for c in paid_tri.columns],
    y=[str(y) for y in paid_tri.index],
    colorscale='Blues',
    text=(paid_tri / 1_000_000).round(1).values,
    texttemplate='%{text:.1f}M',
    textfont={'size': 9},
    hoverongaps=False,
))
fig.update_layout(
    title=f'Triangulo de Perdidas Pagadas — {LOB} ($M)',
    template='plotly_white', height=450, font_family='Georgia',
    yaxis=dict(autorange='reversed'),
)
fig.show()

## 2. Factores Age-to-Age (Link Ratios)

El factor age-to-age mide cuanto crecen las perdidas de un periodo de desarrollo al siguiente:

$$f_{j \to j+1} = \frac{\sum_i C_{i,j+1}}{\sum_i C_{i,j}}$$

Donde $C_{i,j}$ son las perdidas acumuladas del ano de accidente $i$ al lag $j$.

In [ ]:
def compute_ata_factors(triangle):
    """Compute volume-weighted age-to-age factors."""
    lags = sorted(triangle.columns)
    factors = {}
    for j in range(len(lags) - 1):
        curr, nxt = lags[j], lags[j+1]
        mask = triangle[[curr, nxt]].dropna()
        if mask[curr].sum() > 0:
            factors[f'{curr}-{nxt}'] = mask[nxt].sum() / mask[curr].sum()
        else:
            factors[f'{curr}-{nxt}'] = 1.0
    return factors

ata = compute_ata_factors(paid_tri)
print('Age-to-Age Factors (Paid):')
for period, factor in ata.items():
    print(f'  {period}: {factor:.4f}')

In [ ]:
# Plot ATA factors
fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(ata.keys()), y=list(ata.values()),
    marker_color='#1E3A5F',
    text=[f'{v:.4f}' for v in ata.values()],
    textposition='outside',
))
fig.add_hline(y=1.0, line_dash='dash', line_color='#C4841D', annotation_text='1.0 (no development)')
fig.update_layout(
    title=f'Factores Age-to-Age — {LOB} (Pagadas)',
    template='plotly_white', height=350, font_family='Georgia',
    yaxis_title='Factor', xaxis_title='Periodo de Desarrollo',
)
fig.show()

## 3. Chain-Ladder: Proyeccion a Ultimo

El CDF (Cumulative Development Factor) acumula los factores ATA desde cada lag hasta el ultimo:

$$CDF_j = \prod_{k=j}^{n-1} f_{k \to k+1}$$

La perdida ultima se estima como: $Ultimate_i = C_{i,j_i} \times CDF_{j_i}$

Donde $j_i$ es el ultimo lag disponible para el ano de accidente $i$.

In [ ]:
# Cumulative development factors
factor_values = list(ata.values())
lags = sorted(paid_tri.columns)

cdfs = {}
for j in range(len(lags)):
    cdf = 1.0
    for k in range(j, len(lags) - 1):
        cdf *= factor_values[k]
    cdfs[lags[j]] = cdf

print('CDFs to Ultimate:')
for lag, cdf in cdfs.items():
    print(f'  From lag {lag}: {cdf:.4f}')

In [ ]:
# Project ultimates
results = []
for year in sorted(paid_tri.index):
    row = paid_tri.loc[year].dropna()
    if row.empty:
        continue
    latest_lag = row.index.max()
    reported = row[latest_lag]
    cdf = cdfs[latest_lag]
    ultimate = reported * cdf
    ibnr_est = max(0, ultimate - reported)
    results.append({
        'Accident Year': year,
        'Latest Lag': latest_lag,
        'Reported Paid': f'${reported:,.0f}',
        'CDF': f'{cdf:.4f}',
        'Ultimate (CL)': f'${ultimate:,.0f}',
        'IBNR': f'${ibnr_est:,.0f}',
    })

pd.DataFrame(results)

## 4. Comparacion Chain-Ladder vs Bornhuetter-Ferguson

Los resultados pre-calculados incluyen ambos metodos. BF modera las proyecciones usando una loss ratio a-priori.

In [ ]:
lob_ibnr = ibnr[ibnr['line_of_business'] == LOB].copy()

fig = go.Figure()
fig.add_trace(go.Bar(x=lob_ibnr['accident_year'], y=lob_ibnr['ibnr_cl_paid'],
                     name='IBNR (Chain-Ladder)', marker_color='#1E3A5F'))
fig.add_trace(go.Bar(x=lob_ibnr['accident_year'], y=lob_ibnr['ibnr_bf'],
                     name='IBNR (Bornhuetter-Ferguson)', marker_color='#C4841D'))
fig.update_layout(
    title=f'IBNR por Ano de Accidente — {LOB}',
    template='plotly_white', height=400, font_family='Georgia',
    barmode='group', xaxis_title='Ano de Accidente', yaxis_title='IBNR ($)',
)
fig.show()

## 5. Comparacion Across LOBs

In [ ]:
# Total IBNR by LOB
ibnr_by_lob = ibnr.groupby('line_of_business').agg(
    total_ibnr_cl=('ibnr_cl_paid', 'sum'),
    total_ibnr_bf=('ibnr_bf', 'sum'),
    total_reported=('reported_paid', 'sum'),
    total_premium=('earned_premium', 'sum'),
).reset_index()
ibnr_by_lob['ibnr_pct_of_paid'] = (ibnr_by_lob['total_ibnr_cl'] / ibnr_by_lob['total_reported'] * 100).round(1)

fig = px.bar(
    ibnr_by_lob, x='line_of_business', y='total_ibnr_cl',
    color='ibnr_pct_of_paid', color_continuous_scale='Reds',
    labels={'total_ibnr_cl': 'IBNR Total ($)', 'line_of_business': '',
            'ibnr_pct_of_paid': 'IBNR como % de Pagadas'},
    title='IBNR Total por LOB (Chain-Ladder sobre Pagadas)',
)
fig.update_layout(template='plotly_white', height=400, font_family='Georgia')
fig.show()

## So What?

- Los factores de desarrollo pagado son consistentemente > 1 (los pagos siempre aumentan), mientras que los factores de incurrido pueden ser < 1 (desarrollo favorable de reservas de caso).
- Los anos de accidente mas recientes tienen CDFs mas altos porque tienen menos desarrollo observado — mas incertidumbre.
- BF produce estimaciones mas conservadoras que CL para anos con poca informacion, al incorporar la loss ratio a-priori.
- Las lineas de cola larga (med mal, product liability) tienen los CDFs mas altos: se necesitan mas anos de desarrollo antes de conocer las perdidas finales.
- La diferencia CL vs BF es mayor en anos recientes — exactamente donde la incertidumbre es mayor y el juicio actuarial mas valioso.